# End-to-End NDPI Pipeline Demo (In-Process)

This notebook runs the full preprocessing + RF-DETR/YOLO training + annotation pipeline
by importing functions directly from src/ (no subprocess calls).
Update the paths in the next cell to match your data layout.

Steps:
1. Read NDPI+NDPA pairs and generate H5 tiles
2. Postprocess tiles (focus stack + rankings)
3. Split into train/val/test
4. Export COCO (single class, filtered boxes)
5. Train RF-DETR (optional)
6. Train YOLO26 (optional)
7. Hyperparameter tuning (RF-DETR + YOLO26, optional)
8. Run annotator on a new slide

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

# ---- Update these paths ----
repo_root = Path('..')  # notebook lives in notebooks/
input_ndpi_dir = Path('/path/to/ndpi_with_ndpa')  # contains .ndpi + matching .ndpi.ndpa
annotation_map_csv = Path('/path/to/annotation_categories.csv')

# Output roots
h5_dir = repo_root / 'output' / 'tiles_h5'
splits_dir = repo_root / 'output' / 'splits'
coco_dir = repo_root / 'output' / 'coco_export'
rfdetr_out = repo_root / 'output' / 'rfdetr_run'
yolo_out = repo_root / 'output' / 'yolo26_run'
rfdetr_tune_out = repo_root / 'output' / 'rfdetr_tune'
yolo_tune_out = repo_root / 'output' / 'yolo26_tune'
annotator_out = repo_root / 'output' / 'annotator'

# New slide to annotate (not part of train/val/test)
annotate_ndpi_path = Path('/path/to/new_slide.ndpi')

# Hyperparameters (edit as needed)
magnification = 40
tile_size = 1024
overlap = 0.0

rfdetr_model = 'base'
rfdetr_epochs = 60
rfdetr_batch_size = 4
rfdetr_grad_accum = 2
rfdetr_lr = 5e-5
rfdetr_lr_scheduler = 'cosine'
rfdetr_lr_min_factor = 0.1
rfdetr_warmup_epochs = 2
rfdetr_weight_decay = 0.01
rfdetr_imgsz = 1008  # must be divisible by 56 for base/small/nano/large
rfdetr_workers = 6
rfdetr_early_stop = 8
rfdetr_drop_path = 0.1
rfdetr_aug = 'custom'

# YOLO26 training hyperparameters
yolo_model = 'yolo26l.pt'
yolo_epochs = 100
yolo_imgsz = 640
yolo_batch_size = 16
yolo_workers = 6
yolo_optimizer = 'AdamW'
yolo_lr0 = 1e-3
yolo_lrf = 0.1
yolo_patience = 10
yolo_aug = 'custom'
yolo_device = None
yolo_run_name = 'train'
yolo_momentum = None
yolo_weight_decay = None
yolo_warmup_epochs = None
yolo_box = None
yolo_cls_gain = None
yolo_dfl = None

# Hyperparameter tuning (edit as needed)
rfdetr_tune_trials = 8
rfdetr_tune_epochs = 30
rfdetr_tune_method = 'bayesian'
rfdetr_tune_nproc = 1
rfdetr_tune_seed = 67123
rfdetr_tune_aug = False
rfdetr_tune_search_space = {
    'lr': ['log_float', 1e-6, 1e-3],
    'weight_decay': ['log_float', 1e-4, 0.1],
    'drop_path': ['float', 0.0, 0.3],
    'warmup_epochs': ['int', 1, 5],
    'lr_min_factor': ['float', 0.01, 0.3],
}
rfdetr_tune_aug_search_space = {
    'rotate_p': ['float', 0.3, 0.9],
    'brightness_p': ['float', 0.1, 0.5],
    'hsv_p': ['float', 0.1, 0.5],
}

yolo_tune_model = 'l'
yolo_tune_epochs = 30
yolo_tune_imgsz = 640
yolo_tune_batch_size = 16
yolo_tune_workers = 6
yolo_tune_patience = 10
yolo_tune_iterations = 10
yolo_tune_search_alg = 'optuna'
yolo_tune_seed = 67123
yolo_tune_aug = False
yolo_tune_aug_config = 'custom'
yolo_tune_search_space = {
    'lr0': ['log_float', 1e-5, 1e-2],
    'lrf': ['float', 0.01, 0.5],
    'momentum': ['float', 0.7, 0.98],
    'weight_decay': ['float', 0.0, 1e-3],
    'warmup_epochs': ['float', 0.0, 5.0],
    'box': ['float', 1.0, 20.0],
    'cls': ['float', 0.1, 4.0],
    'dfl': ['float', 0.4, 12.0],
}
yolo_tune_aug_search_space = {
    'fliplr': ['float', 0.0, 1.0],
    'flipud': ['float', 0.0, 1.0],
    'degrees': ['float', 0.0, 180.0],
    'hsv_h': ['float', 0.0, 0.1],
    'hsv_s': ['float', 0.0, 0.9],
    'hsv_v': ['float', 0.0, 0.9],
    'translate': ['float', 0.0, 0.3],
    'scale': ['float', 0.0, 0.5],
    'mosaic': ['float', 0.0, 1.0],
    'mixup': ['float', 0.0, 0.3],
    'copy_paste': ['float', 0.0, 1.0],
}

annotator_conf_thresh = 0.5
annotator_nms_iou = 0.5
annotator_magnification = 20
annotator_overlap = 0.10
annotator_compression = 'best_focal_plane'  # or 'focus_stack'
annotator_rfdetr_variant = 'base'

# ---- Imports from src ----
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.preprocessing.generate_tiles import generate_tiles_for_dir
from src.preprocessing.h5_utils import list_h5_paths
from src.preprocessing.postprocess_tiles import process_h5_file
from src.preprocessing.split_data import split_data as split_data_fn
from src.preprocessing.export_coco import ExportMode, run as export_coco_run
from src.annotator.config import AnnotatorConfig
from src.annotator.pipeline import NDPIAnnotator
from src.models.rfdetr.train import (
    _MODEL_CLASSES,
    _MODEL_IMGSZ_DIVISOR,
    AUG_CONFIG,
    _prepare_roboflow_layout,
    _read_class_names,
)

# Basic path checks
assert repo_root.is_dir(), f'Repo not found: {repo_root}'
assert input_ndpi_dir.is_dir(), f'NDPI dir not found: {input_ndpi_dir}'
assert annotation_map_csv.is_file(), f'CSV not found: {annotation_map_csv}'

## 1) Generate H5 tiles from NDPI+NDPA

In [ ]:
h5_dir.mkdir(parents=True, exist_ok=True)

metadata = generate_tiles_for_dir(
    input_dir=str(input_ndpi_dir),
    output_dir=str(h5_dir),
    annotation_map_path=str(annotation_map_csv),
    magnification=magnification,
    tile_size=tile_size,
    overlap=overlap,
 )

print(f"Wrote metadata to {h5_dir / 'metadata.json'}")

## 2) Postprocess tiles (focus stack + rankings)

In [ ]:
h5_paths = list_h5_paths(str(h5_dir))
if not h5_paths:
    raise FileNotFoundError(f'No .h5 files found in {h5_dir}')

for h5_path in h5_paths:
    process_h5_file(
        h5_path=h5_path,
        write_focus_stacked=True,
        write_rankings=True,
        write_mip=False,
    )

## 3) Split slides into train/val/test

In [ ]:
splits_dir.mkdir(parents=True, exist_ok=True)
splits = split_data_fn(str(h5_dir), seed=67)

splits_json = splits_dir / 'train_val_test.json'
with open(splits_json, 'w') as f:
    json.dump(splits, f, indent=2)

print(f'Saved splits to {splits_json}')
print(f"Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

## 4) Export focus-stacked COCO (single class, filtered boxes)

In [ ]:
coco_dir.mkdir(parents=True, exist_ok=True)
mode = ExportMode(kind='focus_stack', plane_indices=[])
export_coco_run(
    h5_root=str(h5_dir),
    output_dir=str(coco_dir),
    mode=mode,
    splits_json=str(splits_json),
    workers=4,
    image_format='jpeg',
    single_cls=True,
    metadata_json=None,
    filter_bboxes=True,
 )

## 5) Train RF-DETR (optional)

In [ ]:
rfdetr_out.mkdir(parents=True, exist_ok=True)
divisor = _MODEL_IMGSZ_DIVISOR[rfdetr_model]
if rfdetr_imgsz % divisor != 0:
    raise ValueError(
        f'--imgsz {rfdetr_imgsz} is not divisible by {divisor} for model {rfdetr_model}.'
    )

staging_dir = rfdetr_out / '.rfdetr_dataset'
_prepare_roboflow_layout(str(coco_dir), str(staging_dir))
class_names = _read_class_names(str(coco_dir))
aug = dict(AUG_CONFIG[rfdetr_aug])

model = _MODEL_CLASSES[rfdetr_model]()
model.train(
    dataset_dir=str(staging_dir),
    epochs=rfdetr_epochs,
    batch_size=rfdetr_batch_size,
    grad_accum_steps=rfdetr_grad_accum,
    lr=rfdetr_lr,
    lr_scheduler=rfdetr_lr_scheduler,
    lr_min_factor=rfdetr_lr_min_factor,
    warmup_epochs=rfdetr_warmup_epochs,
    weight_decay=rfdetr_weight_decay,
    resolution=rfdetr_imgsz,
    output_dir=str(rfdetr_out),
    num_workers=rfdetr_workers,
    class_names=class_names,
    early_stopping=True,
    early_stopping_patience=rfdetr_early_stop,
    early_stopping_use_ema=True,
    aug_config=aug,
    drop_path=rfdetr_drop_path,
 )

## 6) Train YOLO26 (optional)

In [ ]:
from ultralytics import YOLO

from src.models.yolo26.train import build_training_overrides
from src.models.yolo26.utils import (
    convert_coco_labels_to_yolo,
    get_coco_yaml_path,
    validate_coco_dir,
 )

yolo_out.mkdir(parents=True, exist_ok=True)

validate_coco_dir(str(coco_dir))
convert_coco_labels_to_yolo(str(coco_dir))
yaml_path = get_coco_yaml_path(str(coco_dir))

overrides = build_training_overrides(
    yaml_path=yaml_path,
    epochs=yolo_epochs,
    imgsz=yolo_imgsz,
    batch=yolo_batch_size,
    workers=yolo_workers,
    project=str(yolo_out),
    name=yolo_run_name,
    device=yolo_device,
    optimizer=yolo_optimizer,
    lr0=yolo_lr0,
    lrf=yolo_lrf,
    patience=yolo_patience,
    aug_config=yolo_aug,
    momentum=yolo_momentum,
    weight_decay=yolo_weight_decay,
    warmup_epochs=yolo_warmup_epochs,
    box=yolo_box,
    cls_gain=yolo_cls_gain,
    dfl=yolo_dfl,
)

model = YOLO(yolo_model)
model.train(**overrides)

## 7) Hyperparameter tuning (RF-DETR + YOLO26)

These runs can be slow; keep trial counts small for a quick demo.

In [ ]:
from types import SimpleNamespace

from src.models.rfdetr import tune as rfdetr_tune

rfdetr_tune_out.mkdir(parents=True, exist_ok=True)
trials_dir = rfdetr_tune_out / 'trials'
trials_dir.mkdir(parents=True, exist_ok=True)

divisor = _MODEL_IMGSZ_DIVISOR[rfdetr_model]
if rfdetr_imgsz % divisor != 0:
    raise ValueError(
        f'rfdetr_imgsz {rfdetr_imgsz} is not divisible by {divisor} '
        f'for model {rfdetr_model}.'
    )

if rfdetr_tune_aug and not rfdetr_tune_aug_search_space:
    raise ValueError('rfdetr_tune_aug_search_space is required when rfdetr_tune_aug is True.')

rfdetr_tune_args = SimpleNamespace(
    coco_dir=str(coco_dir),
    output_dir=str(rfdetr_tune_out),
    model=rfdetr_model,
    nproc=rfdetr_tune_nproc,
    epochs=rfdetr_tune_epochs,
    imgsz=rfdetr_imgsz,
    workers=rfdetr_workers,
    early_stopping_patience=rfdetr_early_stop,
    method=rfdetr_tune_method,
    n_trials=rfdetr_tune_trials,
    seed=rfdetr_tune_seed,
    tune_aug=rfdetr_tune_aug,
    batch_size=rfdetr_batch_size,
    grad_accum=rfdetr_grad_accum,
    search_space=json.dumps(rfdetr_tune_search_space),
    aug_search_space=(
        json.dumps(rfdetr_tune_aug_search_space)
        if rfdetr_tune_aug
        else None
    ),
)

best_params = rfdetr_tune._tune(rfdetr_tune_args, trials_dir)
best_path = rfdetr_tune_out / 'best_params.json'
with open(best_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Saved best RF-DETR params to {best_path}')

In [ ]:
from types import SimpleNamespace

from src.models.yolo26 import tune as yolo_tune
from src.models.yolo26.utils import (
    convert_coco_labels_to_yolo,
    get_coco_yaml_path,
    validate_coco_dir,
 )

yolo_tune_out.mkdir(parents=True, exist_ok=True)
trials_dir = yolo_tune_out / 'trials'
trials_dir.mkdir(parents=True, exist_ok=True)

validate_coco_dir(str(coco_dir))
if yolo_tune_imgsz % 32 != 0:
    raise ValueError(f'yolo_tune_imgsz {yolo_tune_imgsz} must be divisible by 32.')

if yolo_tune_aug and not yolo_tune_aug_search_space:
    raise ValueError('yolo_tune_aug_search_space is required when yolo_tune_aug is True.')

convert_coco_labels_to_yolo(str(coco_dir))
yaml_path = get_coco_yaml_path(str(coco_dir))

yolo_tune_args = SimpleNamespace(
    coco_dir=str(coco_dir),
    output_dir=str(yolo_tune_out),
    model=yolo_tune_model,
    epochs=yolo_tune_epochs,
    imgsz=yolo_tune_imgsz,
    batch=yolo_tune_batch_size,
    workers=yolo_tune_workers,
    patience=yolo_tune_patience,
    iterations=yolo_tune_iterations,
    search_alg=yolo_tune_search_alg,
    seed=yolo_tune_seed,
    aug_config=yolo_tune_aug_config,
    tune_aug=yolo_tune_aug,
    search_space=json.dumps(yolo_tune_search_space),
    aug_search_space=(
        json.dumps(yolo_tune_aug_search_space)
        if yolo_tune_aug
        else None
    ),
    device=yolo_device,
)

best_params = yolo_tune._tune(yolo_tune_args, yolo_tune_out, trials_dir, yaml_path)
best_path = yolo_tune_out / 'best_params.json'
with open(best_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'Saved best YOLO26 params to {best_path}')

## 8) Run annotator on a new slide

In [ ]:
# Update this to the best checkpoint produced in the run directory
checkpoint_path = rfdetr_out / 'checkpoint_best_ema.pth'
assert checkpoint_path.is_file(), f'Checkpoint not found: {checkpoint_path}'

annotator_out.mkdir(parents=True, exist_ok=True)
config = AnnotatorConfig(
    ndpi_path=str(annotate_ndpi_path),
    output_dir=str(annotator_out),
    model_name='rfdetr',
    checkpoint_path=str(checkpoint_path),
    overlap=annotator_overlap,
    magnification=annotator_magnification,
    confidence_threshold=annotator_conf_thresh,
    nms_iou_threshold=annotator_nms_iou,
    compression_method=annotator_compression,
    rfdetr_variant=annotator_rfdetr_variant,
 )
annotator = NDPIAnnotator(config)
detections = annotator.run()
print(f'Wrote {len(detections)} detections to {annotator_out}')